In [1]:
# imports
import re
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

import optuna
from boruta import BorutaPy

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

import shap
import lightgbm as lgb
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    recall_score,
    make_scorer,
    fbeta_score,
    roc_auc_score
)
from sklearn.model_selection import cross_val_score
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import TunedThresholdClassifierCV

In [2]:
from features import FEATURES

In [3]:
# seed
SEED = 42
TARGET = 'is_fraud'

In [4]:
# train
FILE_TRAIN_TRANSACTION = Path('datasets/train_transaction.csv')
FILE_TRAIN_IDENTITY = Path('datasets/train_identity.csv')

# test
FILE_TEST_TRANSACTION = Path('datasets/test_transaction.csv')
FILE_TEST_IDENTITY = Path('datasets/test_identity.csv')

In [5]:
# leitura eager 
df_train_transaction = pl.read_csv(FILE_TRAIN_TRANSACTION)
df_train_identity = pl.read_csv(FILE_TRAIN_IDENTITY)
## cruza os datasets
df_train = df_train_transaction.join(
    df_train_identity,
    on='TransactionID',
    how='left'
)

# leitura lazy
df_test_transaction = pl.scan_csv(FILE_TEST_TRANSACTION)
df_test_identity = pl.scan_csv(FILE_TEST_IDENTITY)
## cruza os datasets
df_test = df_test_transaction.join(
    df_test_identity,
    on='TransactionID',
    how='left'
)

In [6]:
# normalize columns
def normalize_cols(col):
    col = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', col)
    col = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', col)
    return col.lower()

In [7]:
df_train = df_train.rename({col: normalize_cols(col) for col in df_train.columns})
# df_test = df_test.rename({col: normalize_cols(col) for col in df_test.collect_schema().names()})

In [8]:
cat_cols = df_train.select(FEATURES).columns

In [9]:
# optuna
def objective(trial):
    params = {
        # Capacidade da árvore
        'num_leaves': trial.suggest_int('num_leaves', 10, 100),
        'max_depth': trial.suggest_int('max_depth', 2, 10),

        # Regularização
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-4, 1e-1, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),

        # Amostragem
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'subsample_freq': trial.suggest_int('subsample_freq', 1, 10),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),

        # Boosting
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 2000),

        # Fixos
        'random_state': SEED,
        'n_jobs': 2,
        'verbose': -1,
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 20.0),
    }

    clf = lgb.LGBMClassifier(**params)
    score = cross_val_score(clf, X_train, y_train, cv=3, scoring="roc_auc").mean()
    return score

In [10]:
study = optuna.create_study(
    storage='sqlite:///optuna_study.db',
    direction="maximize"
)

[I 2026-05-25 18:46:13,699] A new study created in RDB with name: no-name-7f5a3208-6a26-4b34-a0af-acfef0b4ba35


In [11]:
df_train, df_eval = train_test_split(df_train, test_size=0.15, random_state=SEED)

In [12]:
meta_cols = ['transaction_id', 'is_fraud']

In [13]:
X_train, y_train = df_train.select(FEATURES), df_train['is_fraud']
X_eval, y_eval = df_eval.select(FEATURES), df_eval['is_fraud']

In [14]:
print(X_train)

shape: (501_959, 67)
┌─────┬─────────────────┬────────────────┬─────┬───┬───────┬──────┬──────┬─────┐
│ c5  ┆ transaction_amt ┆ transaction_dt ┆ c6  ┆ … ┆ v315  ┆ v49  ┆ d14  ┆ c12 │
│ --- ┆ ---             ┆ ---            ┆ --- ┆   ┆ ---   ┆ ---  ┆ ---  ┆ --- │
│ f64 ┆ f64             ┆ i64            ┆ f64 ┆   ┆ f64   ┆ f64  ┆ f64  ┆ f64 │
╞═════╪═════════════════╪════════════════╪═════╪═══╪═══════╪══════╪══════╪═════╡
│ 0.0 ┆ 445.0           ┆ 9911916        ┆ 2.0 ┆ … ┆ 0.0   ┆ 0.0  ┆ null ┆ 0.0 │
│ 0.0 ┆ 171.0           ┆ 9672822        ┆ 1.0 ┆ … ┆ 0.0   ┆ 0.0  ┆ null ┆ 0.0 │
│ 0.0 ┆ 29.0            ┆ 13022888       ┆ 2.0 ┆ … ┆ 15.0  ┆ null ┆ null ┆ 1.0 │
│ 0.0 ┆ 335.0           ┆ 14784078       ┆ 2.0 ┆ … ┆ 226.0 ┆ 0.0  ┆ null ┆ 0.0 │
│ 0.0 ┆ 44.0            ┆ 12584234       ┆ 1.0 ┆ … ┆ 0.0   ┆ 0.0  ┆ null ┆ 0.0 │
│ …   ┆ …               ┆ …              ┆ …   ┆ … ┆ …     ┆ …    ┆ …    ┆ …   │
│ 0.0 ┆ 25.0            ┆ 2157580        ┆ 1.0 ┆ … ┆ 0.0   ┆ null ┆ null ┆ 0.0 │
│ 1.0 ┆

In [15]:
print(X_eval)

shape: (88_581, 67)
┌──────┬─────────────────┬────────────────┬─────┬───┬───────────┬─────┬──────┬─────┐
│ c5   ┆ transaction_amt ┆ transaction_dt ┆ c6  ┆ … ┆ v315      ┆ v49 ┆ d14  ┆ c12 │
│ ---  ┆ ---             ┆ ---            ┆ --- ┆   ┆ ---       ┆ --- ┆ ---  ┆ --- │
│ f64  ┆ f64             ┆ i64            ┆ f64 ┆   ┆ f64       ┆ f64 ┆ f64  ┆ f64 │
╞══════╪═════════════════╪════════════════╪═════╪═══╪═══════════╪═════╪══════╪═════╡
│ 0.0  ┆ 724.0           ┆ 12153579       ┆ 1.0 ┆ … ┆ 0.0       ┆ 0.0 ┆ null ┆ 0.0 │
│ 0.0  ┆ 108.5           ┆ 15005886       ┆ 1.0 ┆ … ┆ 0.0       ┆ 0.0 ┆ null ┆ 1.0 │
│ 2.0  ┆ 47.95           ┆ 6970178        ┆ 1.0 ┆ … ┆ 87.949997 ┆ 1.0 ┆ null ┆ 0.0 │
│ 0.0  ┆ 100.599         ┆ 5673658        ┆ 1.0 ┆ … ┆ 0.0       ┆ 0.0 ┆ null ┆ 1.0 │
│ 11.0 ┆ 107.95          ┆ 6886780        ┆ 8.0 ┆ … ┆ 0.0       ┆ 1.0 ┆ null ┆ 0.0 │
│ …    ┆ …               ┆ …              ┆ …   ┆ … ┆ …         ┆ …   ┆ …    ┆ …   │
│ 0.0  ┆ 29.183          ┆ 7917448        ┆ 1

In [16]:
X_train, y_train = X_train.to_pandas(), y_train.to_pandas()
X_eval, y_eval = X_eval.to_pandas(), y_eval.to_pandas()

In [17]:
X_train[cat_cols] = X_train[cat_cols].astype('category')
X_eval[cat_cols] = X_eval[cat_cols].astype('category')

In [ ]:
study.optimize(objective, n_trials=50)

[I 2026-05-25 19:02:37,232] Trial 0 finished with value: 0.9538631812751537 and parameters: {'num_leaves': 96, 'max_depth': 8, 'min_child_samples': 23, 'min_child_weight': 0.05149769643989256, 'reg_alpha': 1.7387785185843152, 'reg_lambda': 4.580748140338931, 'subsample': 0.9165834672938717, 'subsample_freq': 9, 'colsample_bytree': 0.7088853575235082, 'learning_rate': 0.210756503343119, 'n_estimators': 1676, 'scale_pos_weight': 17.43220110915287}. Best is trial 0 with value: 0.9538631812751537.


In [ ]:
PARAMS = study.best_params
print(PARAMS)

In [ ]:
dtrain = lgb.Dataset(X_train, label=y_train)
dval = lgb.Dataset(X_eval, label=y_eval, reference=dtrain)

In [ ]:
cv_results = lgb.cv(
    PARAMS,
    dtrain,
    metrics='auc',
    nfold=5,
    stratified=True,
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(100),
    ],
    return_cvbooster=False,
)

In [ ]:
best_rounds = len(cv_results['valid auc-mean'])
best_auc = cv_results['valid auc-mean'][-1]
best_std = cv_results['valid auc-stdv'][-1]
print(f"Best AUC: {best_auc:.4f} ± {best_std:.4f} | Rounds: {best_rounds}")

In [ ]:
model = lgb.train(
    params,
    dtrain,
    num_boost_round=best_rounds,
    valid_sets=[dtrain, dval],
    valid_names=['train', 'val'],
    callbacks=[lgb.log_evaluation(100)],
)

In [ ]:
y_pred = model.predict(X_eval)

In [ ]:
print(classification_report(y_eval, y_pred))

In [ ]:
print(confusion_matrix(y_eval, y_pred))

In [ ]:
y_proba = model.predict_proba(X_train)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_train, y_proba)

In [ ]:
best_idx = np.argmax(recall)
best_threshold = thresholds[best_idx]

In [ ]:
f1 = 2 * (precision * recall) / (precision + recall + 1e-8)

best_idx = np.argmax(f1)
best_threshold = thresholds[best_idx]

In [ ]:
best_threshold.item()

In [ ]:
import matplotlib.pyplot as plt

plt.plot(thresholds, recall[:-1], label="Recall")
plt.plot(thresholds, precision[:-1], label="Precision")
plt.legend()
plt.xlabel("Threshold")
plt.show()

In [ ]:
y_proba = model.predict_proba(X_eval)[:, 1]

In [ ]:
y_pred = np.where(y_proba > best_threshold, 1, 0)

In [ ]:
print(classification_report(y_eval, y_pred))

In [ ]:
print(confusion_matrix(y_eval, y_pred))

In [ ]:
tuned_classifier = TunedThresholdClassifierCV(
    model, cv=5, scoring=make_scorer(roc_auc_score)
).fit(X_train, y_train)

In [ ]:
tuned_classifier.best_threshold_.item()

In [ ]:
print(f"Custom score: {recall_score(y_eval, tuned_classifier.predict(X_eval)):.2f}")

In [ ]:
print(classification_report(y_eval, tuned_classifier.predict(X_eval)))

In [ ]:
print(confusion_matrix(y_eval, tuned_classifier.predict(X_eval)))

# Submition

In [ ]:
df_test = df_test.collect()

In [ ]:
print(df_test.head())

In [ ]:
'is_fraud' in df_test.columns

In [ ]:
df_test = df_test.rename({col: normalize_cols(col) for col in df_test.columns})

In [ ]:
columns = [x.replace('-', '_') for x in df_test.columns]
df_test.columns = columns

In [ ]:
df_test = df_test.select(['transaction_id'] + FEATURES)

In [ ]:
df_test = df_test.to_pandas()

In [ ]:
print(df_test)

In [ ]:
df_test[cat_cols] = df_test[cat_cols].astype('category')

In [ ]:
X_test = df_test.set_index('transaction_id')

In [ ]:
print(X_test)

In [ ]:
y_pred = tuned_classifier.predict_proba(X_test)[:, 1]

In [ ]:
X_test.index

In [ ]:
# construindo a tabela de submissão
data = {
    'TransactionID': X_test.index,
    'isFraud': y_pred
}
df_submit = pd.DataFrame(data)

In [ ]:
df_submit